## Étape 4.1 : lecture des données

Le CSV est lu depuis HDFS. On vérifie le nombre de colonnes (24 attendues) et de lignes (environ 200 000 attendues).

In [ ]:
import os
from pyspark.sql import SparkSession

DATA_PATH = os.environ.get(
    "DATA_PATH",
    "hdfs://localhost:9000/user/miguel/projet_3IABD/Bank_Transaction_Fraud_Detection.csv",
)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

spark = SparkSession.builder.appName("fraude").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Lecture du CSV depuis HDFS.
raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

# Contrôles : 24 colonnes attendues, environ 200 000 lignes attendues
print("Nombre de colonnes :", len(raw.columns))
print("Nombre de lignes :", raw.count())

raw.printSchema()

26/11/17 15:53:44 WARN Utils: Your hostname, LAPTOP-2O6PEQV6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/17 15:53:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 15:53:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Nombre de colonnes : 24
Nombre de lignes : 200000
root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: string (nullable = true)
 |-- Transaction_Amount: string (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: string (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: string (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Cus

## Étape 4.2 : renommage et typage des colonnes

Les colonnes sont renommées en français, puis typées : les montants, l'âge et la cible en nombres, la date en type date, le reste en texte.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Correspondance entre les noms
RENOMMAGE = {
    "Customer_ID": "id_client",
    "Customer_Name": "nom_client",
    "Gender": "genre",
    "Age": "age",
    "State": "etat",
    "City": "ville",
    "Bank_Branch": "agence_bancaire",
    "Account_Type": "type_compte",
    "Transaction_ID": "id_transaction",
    "Transaction_Date": "date_transaction",
    "Transaction_Time": "heure_transaction",
    "Transaction_Amount": "montant",
    "Merchant_ID": "id_marchand",
    "Transaction_Type": "type_transaction",
    "Merchant_Category": "categorie_marchand",
    "Account_Balance": "solde_compte",
    "Transaction_Device": "appareil_transaction",
    "Transaction_Location": "localisation_transaction",
    "Device_Type": "type_appareil",
    "Is_Fraud": "est_fraude",
    "Transaction_Currency": "devise",
    "Customer_Contact": "contact_client",
    "Transaction_Description": "description_transaction",
    "Customer_Email": "email_client",
}

# Contrôle : toutes les colonnes attendues doivent exister dans le fichier.
manquantes = [c for c in RENOMMAGE if c not in raw.columns]
if manquantes:
    raise ValueError(f"Colonnes absentes du fichier : {manquantes}")


df = raw.select([F.col(c).alias(RENOMMAGE[c]) for c in RENOMMAGE])

# Typage des colonnes
df = (
    df
    .withColumn("age", F.col("age").cast(IntegerType()))
    .withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
    .withColumn("montant", F.col("montant").cast(DoubleType()))
    .withColumn("solde_compte", F.col("solde_compte").cast(DoubleType()))
    .withColumn("date_transaction", F.to_date(F.col("date_transaction"), "dd-MM-yyyy"))
    .withColumn("heure_transaction", F.substring(F.col("heure_transaction"), 1, 2).cast(IntegerType()))
)

# Contrôle : une valeur qui ne se convertit pas devient null.
for c in ["age", "est_fraude", "montant", "solde_compte", "date_transaction"]:
    n = df.filter(F.col(c).isNull()).count()
    print(f"{c} : {n} valeur(s) non convertie(s)")

df.printSchema()

age : 0 valeur(s) non convertie(s)
est_fraude : 0 valeur(s) non convertie(s)
montant : 0 valeur(s) non convertie(s)
solde_compte : 0 valeur(s) non convertie(s)
date_transaction : 0 valeur(s) non convertie(s)
root
 |-- id_client: string (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- etat: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- agence_bancaire: string (nullable = true)
 |-- type_compte: string (nullable = true)
 |-- id_transaction: string (nullable = true)
 |-- date_transaction: date (nullable = true)
 |-- heure_transaction: string (nullable = true)
 |-- montant: double (nullable = true)
 |-- id_marchand: string (nullable = true)
 |-- type_transaction: string (nullable = true)
 |-- categorie_marchand: string (nullable = true)
 |-- solde_compte: double (nullable = true)
 |-- appareil_transaction: string (nullable = true)
 |-- localisation_transaction: string (nullable = t

## Étape 4.3 : statistiques descriptives

On décrit les colonnes numériques, la répartition de la cible `est_fraude`, puis les liens entre les variables et la fraude. Lors d'une analyse préalable sur le fichier complet, le taux de fraude était d'environ 5 % et les corrélations proches de 0. Ces chiffres devront être retrouvés ici, et interprétés après exécution.

In [ ]:
from pyspark.sql import functions as F

colonnes_num = ["age", "montant", "solde_compte", "est_fraude"]

df.select(colonnes_num).describe().show()

# 1. Nombre de transactions normales et frauduleuses
total = df.count()
(
    df.groupBy("est_fraude").count()
    .withColumn("taux_%", F.round(F.col("count") / total * 100, 2))
    .orderBy("est_fraude")
    .show()
)

# 2.Montant moyen par classe

df.groupBy("est_fraude").agg(F.round(F.avg("montant"), 2).alias("montant_moyen")).show()

# 3. Corrélation de chaque variable numérique avec la cible.
for c in ["age", "montant", "solde_compte"]:
    r = df.stat.corr(c, "est_fraude")
    print(f"corrélation {c} / est_fraude : {r:.4f}")

#4. Taux de fraude par catégorie.
for c in ["type_transaction", "categorie_marchand", "type_appareil"]:
    (
        df.groupBy(c)
        .agg(F.round(F.avg("est_fraude") * 100, 2).alias("taux_fraude_%"))
        .orderBy(F.desc("taux_fraude_%"))
        .show()
    )

# 5. Export des statistiques descriptives
stats_pd = df.select(colonnes_num).describe().toPandas()
stats_pd.to_csv(f"{OUT}/statistiques.csv", index=False)
print("Statistiques exportées dans", f"{OUT}/statistiques.csv")

26/11/17 15:53:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+-------------------+
|summary|               age|           montant|      solde_compte|         est_fraude|
+-------+------------------+------------------+------------------+-------------------+
|  count|            200000|            200000|            200000|             200000|
|   mean|          44.01511|49538.015554200014| 52437.98878355006|            0.05044|
| stddev|15.288774326969515| 28551.87400382295|27399.507128068493|0.21885165267877088|
|    min|                18|             10.29|           5000.82|                  0|
|    max|                70|          98999.98|          99999.95|                  1|
+-------+------------------+------------------+------------------+-------------------+

+----------+------+------+
|est_fraude| count|taux_%|
+----------+------+------+
|         0|189912| 94.96|
|         1| 10088|  5.04|
+----------+------+------+

+----------+-------------+
|est_fraude|montant_moyen|

## Étape 4.4 : nettoyage

Le nettoyage porte sur cinq points : suppression de la colonne constante `devise`, retrait des données personnelles, doublons sur l'identifiant de transaction, valeurs manquantes, et plages de valeurs. Lors de l'analyse préalable, aucun doublon ni valeur manquante n'a été trouvé : la cellule le vérifie et le documente.

In [ ]:
from pyspark.sql import functions as F

# 1. Colonne constante
nb_devises = df.select("devise").distinct().count()
print("Valeurs distinctes de devise :", nb_devises)
df = df.drop("devise")

# 2. Données personnelles
df = df.drop("nom_client", "contact_client", "email_client", "id_client", "id_transaction", "id_marchand")

# 3. Doublons
n_avant = df.count()
df = df.dropDuplicates(["id_transaction"])
print("Doublons supprimés :", n_avant - df.count())

# 4. Valeurs manquantes
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

# Suppression des lignes incomplètes
n_avant = df.count()
df = df.dropna()
print("Lignes incomplètes supprimées :", n_avant - df.count())

# 5. Retire les lignes incohérentes
n_avant = df.count()
df = df.filter(
    F.col("age").between(18, 120)
    & (F.col("montant") > 0)
    & (F.col("solde_compte") >= 0)
    & F.col("est_fraude").isin(0, 1)
)
print("Lignes incohérentes supprimées :", n_avant - df.count())

# Nombre de lignes et de colonnes conservées
print("Lignes après nettoyage :", df.count())
print("Colonnes après nettoyage :", len(df.columns))

Valeurs distinctes de devise : 1


Doublons supprimés : 0


+---------+-----+---+----+-----+---------------+-----------+--------------+----------------+-----------------+-------+-----------+----------------+------------------+------------+--------------------+------------------------+-------------+----------+-----------------------+
|id_client|genre|age|etat|ville|agence_bancaire|type_compte|id_transaction|date_transaction|heure_transaction|montant|id_marchand|type_transaction|categorie_marchand|solde_compte|appareil_transaction|localisation_transaction|type_appareil|est_fraude|description_transaction|
+---------+-----+---+----+-----+---------------+-----------+--------------+----------------+-----------------+-------+-----------+----------------+------------------+------------+--------------------+------------------------+-------------+----------+-----------------------+
|0        |0    |0  |0   |0    |0              |0          |0             |0               |0                |0      |0          |0               |0                 |0        

Lignes incomplètes supprimées : 0


Lignes incohérentes supprimées : 0


Lignes après nettoyage : 200000
Colonnes après nettoyage : 20
